In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"



In [0]:
%sql
SHOW VOLUMES IN voebem.bronze;


In [0]:
bruto = (
    spark.read.format("csv")
    .option("header", True)
    .option("sep", ";")
    .option("skipRows", 1)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")  
    .option("mode", "PERMISSIVE") #camada de bronze não descarta linhas, aceita tudo que chega
    .load(CAMINHO)
)

print("Colunas lidas do arquivo: ")
for c in bruto.columns:
    print(f"{c!r}")

Colunas lidas do arquivo: 
'ICAO Empresa Aérea'
'Número Voo'
'Código Autorização (DI)'
'Código Tipo Linha'
'ICAO Aeródromo Origem'
'ICAO Aeródromo Destino'
'Partida Prevista'
'Partida Real'
'Chegada Prevista'
'Chegada Real'
'Situação Voo'
'Código Justificativa'


In [0]:
RENOMEAR = { "ICAO Empresa Aérea": "icao_empresa_aerea",
"Número Voo": "numero_voo",
"Código Autorização (DI)": "codigo_autorizacao_di",
"Código Tipo Linha": "codigo_tipo_linha",
"ICAO Aeródromo Origem": "icao_aerodromo_origem",
"ICAO Aeródromo Destino": "icao_aerodromo_destino",
"Partida Prevista": "partida_prevista",
"Partida Real": "partida_real",
"Chegada Prevista": "chegada_prevista",
"Chegada Real": "chegada_real",
"Situação Voo": "situacao_voo",
"Código Justificativa": "codigo_justificativa"
}

corrigindo = [c for c in RENOMEAR if c not in bruto.columns]
assert not corrigindo, f"Colunas não encontradas: {corrigindo}"

renomeando = bruto.select(
    *[F.col(f"`{origem}`").cast("string").alias(destino) for origem, destino in RENOMEAR.items()]
)

In [0]:
# Para rastreabilidade: como estava e quando veio
bronze = renomeando.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")).withColumn(
        "_ingerido_em", F.current_timestamp()
    )

In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TABELA)
)

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")

voebem.bronze.vra: 1,597,255 linhas


In [0]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS
          'Bronze - VRA (Voo Regular Ativo) da ANAC, 19 meses (jan/2025 a jul/2026).
          Dado bruto: todas as colunas string, nenhuma linha descartada.
          Carga Full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/*.'
        """)

DataFrame[]

In [0]:
display(
    spark.sql(f"""
              SELECT _arquivo_origem, COUNT(*) AS linhas, MAX (_ingerido_em) AS ingerido_em
              FROM {TABELA}
              GROUP BY _arquivo_origem
              ORDER BY _arquivo_origem
              """)
)

_arquivo_origem,linhas,ingerido_em
VRA_20251.csv,89616,2026-09-17T22:13:14.882Z
VRA_202510.csv,85709,2026-09-17T22:13:14.882Z
VRA_202511.csv,82245,2026-09-17T22:13:14.882Z
VRA_202512.csv,88564,2026-09-17T22:13:14.882Z
VRA_20252.csv,78930,2026-09-17T22:13:14.882Z
VRA_20253.csv,82048,2026-09-17T22:13:14.882Z
VRA_20254.csv,80692,2026-09-17T22:13:14.882Z
VRA_20255.csv,82565,2026-09-17T22:13:14.882Z
VRA_20256.csv,80974,2026-09-17T22:13:14.882Z
VRA_20257.csv,87725,2026-09-17T22:13:14.882Z
